# VACUUM
Comando sql que elimina archivos de datos obsoletos de una tabla delta lake

- **libera espacio**

        Elimina archivos no referenciados por el transaccion log
- **Mantiene la seguridad**

        Respeta un periodo de retencion configurable (por defecto 7  dias)
- **Solo delta lake**

        Diseñado especificamente para tablas con formato delta lake 

# Operaciones delta lake y archivos inmutables
Cada operacion genera nuevos archivos parquet

- **insert:** crea nuevos archivos parquet
- **update:** reescribe archivos modificados
- **delete:** marca filas como eliminadas
- **merge:** combina inserts y updates

## Los archivos parquet son inmutables

es decir nunca se modifican ni se eliminan automaticamente, por este motivo los archivos obsoletos se acumulan en el almacenamiento

# ¿que pasa cuando ejecutamos el comendo VACUUM?

## que elimina

- Archivos parquet no referenciados por el transaction log.
- Archivos mas antiguos que el periodo de retencion.
- Datos de versiones antiguas de la tabla.
- Archivos temporales de operacion fallidas.

# Sintaxis basica de VACUUM

In [0]:
%sql
-- Limpiar archivos obsoletos de una tabla
VACUUM workspace.default.users;

# Periodo de retencion por defecto (7 dias)

**¿que significa?**
- Hoy se ejecuta el comando VACUUM.
- Ultimos 7 dias -> archivos protegidos - Time Travel disponoble.
- Mas de dias -> Archivos ELIMINADOS por VACUUM.

# VACUUM con retencion personalizada

_reducir la retencion por debajo de 7 dias puede impedir el Time Travel y la recuperacion de datos_

**Comando para deshabilitar la verificacion de retencion**

`SET spark.databricks.delta.retentionDurationCheck.enabled = false;`

**Ejecutar VACUUM con retencion personalizada**

Retener solo las ultimas 48 horas

`VACUUM users RETAIN 48 HOURS;`

_Valor minimo recomendado para entornos de produccion es de 7 dias_


# DROP TABLE

Elimina la tabla completa - comportamiento diferente segun la tabla

In [0]:
%sql
DESCRIBE HISTORY users

# MANAGED TABLE

databricks controla tanto los metadatos como los datos fisicos.

- **Sin LOCATION** -> Databricks elige donde guardar.
- **DROP TABLE** -> elimina metadatos y datos fisicos.
- **UNDROP TABLE** -> permite recuperar la tabla

In [0]:
%sql

-- Managed Table (sin LOCATION)
CREATE TABLE ventas USING DELTA;

In [0]:
%sql
-- Eliminar tabla
DROP TABLE ventas

In [0]:
%sql
-- Recuperar tabla
UNDROP TABLE ventas


## External table con LOCATION

`CREATE TABLE ventas
USING DELTA
LOCATION 'S3://bucket/ventas';`